In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

data = pd.read_csv(url, sep="\t", header=None,
                   names=["Label", "Message"])

print(data.head())

  Label                                            Message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [2]:
print("Dataset shape:", data.shape)

print("\nMissing values:")
print(data.isnull().sum())

print("\nSpam and Ham counts:")
print(data["Label"].value_counts())

print("\nPercentage:")
print(data["Label"].value_counts(normalize=True) * 100)

Dataset shape: (5572, 2)

Missing values:
Label      0
Message    0
dtype: int64

Spam and Ham counts:
Label
ham     4825
spam     747
Name: count, dtype: int64

Percentage:
Label
ham     86.593683
spam    13.406317
Name: proportion, dtype: float64


In [3]:
import re
import nltk

from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def clean_text(message):
    message = message.lower()
    message = re.sub(r'[^a-z\s]', '', message)
    words = message.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

data['Clean_Message'] = data['Message'].apply(clean_text)

print(data[['Message', 'Clean_Message']].head())

                                             Message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                       Clean_Message  
0  go jurong point crazy available bugis n great ...  
1                            ok lar joking wif u oni  
2  free entry wkly comp win fa cup final tkts st ...  
3                u dun say early hor u c already say  
4        nah dont think goes usf lives around though  


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X = tfidf.fit_transform(data['Clean_Message'])
y = data['Label']

print("TF-IDF shape:", X.shape)
print("Target shape:", y.shape)

TF-IDF shape: (5572, 8480)
Target shape: (5572,)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training messages:", X_train.shape[0])
print("Testing messages:", X_test.shape[0])

Training messages: 4457
Testing messages: 1115


In [6]:
from sklearn.naive_bayes import MultinomialNB

model_nb = MultinomialNB()
model_nb.fit(X_train, y_train)

y_pred_nb = model_nb.predict(X_test)

print("Naive Bayes model trained successfully!")
print("First 10 predictions:")
print(y_pred_nb[:10])

Naive Bayes model trained successfully!
First 10 predictions:
['ham' 'ham' 'ham' 'spam' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham']


In [7]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("Precision:", precision_score(y_test, y_pred_nb, pos_label="spam"))
print("Recall:", recall_score(y_test, y_pred_nb, pos_label="spam"))
print("F1-score:", f1_score(y_test, y_pred_nb, pos_label="spam"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb, labels=["ham", "spam"]))

Accuracy: 0.9632286995515695
Precision: 1.0
Recall: 0.7248322147651006
F1-score: 0.8404669260700389

Classification Report:
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       966
        spam       1.00      0.72      0.84       149

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115

Confusion Matrix:
[[966   0]
 [ 41 108]]


In [8]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000)

model_lr.fit(X_train, y_train)

y_pred_lr = model_lr.predict(X_test)

print("Logistic Regression model trained successfully!")
print("First 10 predictions:")
print(y_pred_lr[:10])

Logistic Regression model trained successfully!
First 10 predictions:
['ham' 'ham' 'ham' 'spam' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham']


In [9]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr, pos_label="spam"))
print("Recall:", recall_score(y_test, y_pred_lr, pos_label="spam"))
print("F1-score:", f1_score(y_test, y_pred_lr, pos_label="spam"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr, labels=["ham", "spam"]))


Accuracy: 0.9587443946188341
Precision: 1.0
Recall: 0.6912751677852349
F1-score: 0.8174603174603174

Classification Report:
              precision    recall  f1-score   support

         ham       0.95      1.00      0.98       966
        spam       1.00      0.69      0.82       149

    accuracy                           0.96      1115
   macro avg       0.98      0.85      0.90      1115
weighted avg       0.96      0.96      0.96      1115

Confusion Matrix:
[[966   0]
 [ 46 103]]


In [10]:
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Naive Bayes": [
        accuracy_score(y_test, y_pred_nb),
        precision_score(y_test, y_pred_nb, pos_label="spam"),
        recall_score(y_test, y_pred_nb, pos_label="spam"),
        f1_score(y_test, y_pred_nb, pos_label="spam")
    ],
    "Logistic Regression": [
        accuracy_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_lr, pos_label="spam"),
        recall_score(y_test, y_pred_lr, pos_label="spam"),
        f1_score(y_test, y_pred_lr, pos_label="spam")
    ]
})

print(comparison.round(4))

      Metric  Naive Bayes  Logistic Regression
0   Accuracy       0.9632               0.9587
1  Precision       1.0000               1.0000
2     Recall       0.7248               0.6913
3   F1-score       0.8405               0.8175


### What is TF-IDF?

TF-IDF stands for Term Frequency–Inverse Document Frequency. It is a technique used to convert text messages into numerical values that a machine-learning model can understand.

Term Frequency measures how often a word appears in a message. Inverse Document Frequency gives less importance to words that appear in many messages and helps identify more informative words.

In this project, TF-IDF converts SMS messages into numerical features that help the models classify messages as spam or ham.


What is TF-IDF?
TF-IDF stands for Term Frequency–Inverse Document Frequency. It is a technique used to convert text messages into numerical values that a machine-learning model can understand.

Term Frequency measures how often a word appears in a message. Inverse Document Frequency gives less importance to words that appear in many messages and helps identify more informative words.

In this project, TF-IDF converts SMS messages into numerical features that help the models classify messages as spam or ham.


### Why Is Recall Important in Spam Detection?

Recall measures how many actual spam messages are correctly identified by the model.

Recall is important because missed spam messages may reach the user's inbox. A model with higher recall can identify more spam messages and reduce the number of missed spam messages.

In this project, Multinomial Naive Bayes achieved a recall of 72.48%, while Logistic Regression achieved 69.13%. Therefore, Naive Bayes identified more spam messages in our test dataset. However, both models can still be improved.

In [11]:
message = input("Enter a message to check: ")

cleaned_message = clean_text(message)
message_vector = tfidf.transform([cleaned_message])

prediction = model_nb.predict(message_vector)[0]

print("Prediction:", prediction.upper())

Enter a message to check: Congratulations! You have won a free prize. Claim it now!
Prediction: SPAM
